# NHANES dataset inspection

Pulls the NHANES 2017–2018 (cycle **J**) public-use files we care about for the almond risk model and gives a first look at shapes, dtypes, missingness, and key distributions. Also pulls the Public-Use Linked Mortality File so we can sanity-check the survival outcome we'll train on.

Sources:
- NHANES data files (post-2024 CDC reorg): `https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/<start_year>/DataFiles/<CODE>.xpt`
- Public-use linked mortality: `https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/`

Files land in `inspect/data/` (gitignored). Re-running cells skips files already downloaded.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
print("data dir:", DATA_DIR)

data dir: /Users/deniz/Documents/Projects/almond/inspect/data


## 1. Pick which NHANES files to pull

Cycle J = NHANES 2017–2018. These map to the features the Cox model + clinical risk equations will need.

| File | What it gives us |
|---|---|
| `DEMO_J` | age, sex, race/ethnicity, sample weights |
| `BMX_J` | height, weight, BMI, waist circumference |
| `BPX_J` | measured systolic / diastolic BP |
| `BPQ_J` | self-reported hypertension + BP meds |
| `TCHOL_J` | total cholesterol |
| `HDL_J` | HDL cholesterol |
| `DIQ_J` | self-reported diabetes |
| `GHB_J` | HbA1c (glycohemoglobin) |
| `SMQ_J` | smoking history |
| `MCQ_J` | medical conditions (incl. prior CVD) |
| `PAQ_J` | physical activity questionnaire |

In [2]:
# As of 2024 CDC reorganized NHANES hosting. Current pattern:
#   https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/<start_year>/DataFiles/<CODE>.xpt
# (lowercase .xpt; directory is the cycle's start year, e.g. 2017 for 2017-2018.)
@dataclass(frozen=True)
class NhanesFile:
    code: str        # e.g. "DEMO_J"
    start_year: int  # e.g. 2017 for cycle 2017-2018
    label: str

    @property
    def filename(self) -> str:
        return f"{self.code}.xpt"

    @property
    def url(self) -> str:
        return (
            f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{self.start_year}/"
            f"DataFiles/{self.filename}"
        )

    @property
    def local_path(self) -> Path:
        return DATA_DIR / self.filename


FILES: list[NhanesFile] = [
    NhanesFile("DEMO_J",  2017, "Demographics"),
    NhanesFile("BMX_J",   2017, "Body measurements"),
    NhanesFile("BPX_J",   2017, "Blood pressure (measured)"),
    NhanesFile("BPQ_J",   2017, "Blood pressure questionnaire"),
    NhanesFile("TCHOL_J", 2017, "Total cholesterol"),
    NhanesFile("HDL_J",   2017, "HDL cholesterol"),
    NhanesFile("DIQ_J",   2017, "Diabetes questionnaire"),
    NhanesFile("GHB_J",   2017, "HbA1c (glycohemoglobin)"),
    NhanesFile("SMQ_J",   2017, "Smoking questionnaire"),
    NhanesFile("MCQ_J",   2017, "Medical conditions"),
    NhanesFile("PAQ_J",   2017, "Physical activity"),
]
len(FILES), [f.code for f in FILES]

(11,
 ['DEMO_J',
  'BMX_J',
  'BPX_J',
  'BPQ_J',
  'TCHOL_J',
  'HDL_J',
  'DIQ_J',
  'GHB_J',
  'SMQ_J',
  'MCQ_J',
  'PAQ_J'])

## 2. Download (resumable, idempotent)

Skips anything already in `data/`. Each XPT is ~1–10 MB.

In [3]:
def download(url: str, dest: Path, chunk_size: int = 1 << 15) -> Path:
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        ctype = r.headers.get("content-type", "")
        if "html" in ctype.lower():
            raise RuntimeError(f"unexpected HTML response from {url} (likely 404 served as page)")
        total = int(r.headers.get("content-length") or 0) or None
        with tmp.open("wb") as f, tqdm(
            total=total, unit="B", unit_scale=True, desc=dest.name, leave=False
        ) as bar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
    tmp.rename(dest)
    return dest


for f in FILES:
    path = download(f.url, f.local_path)
    size_kb = path.stat().st_size / 1024
    print(f"{f.code:8s}  {size_kb:8.1f} KB  ({f.label})")

DEMO_J      3332.7 KB  (Demographics)
BMX_J       1431.6 KB  (Body measurements)
BPX_J       1431.6 KB  (Blood pressure (measured))
BPQ_J        531.8 KB  (Blood pressure questionnaire)
TCHOL_J      175.5 KB  (Total cholesterol)
HDL_J        175.5 KB  (HDL cholesterol)
DIQ_J       3761.6 KB  (Diabetes questionnaire)
GHB_J        101.1 KB  (HbA1c (glycohemoglobin))
SMQ_J       2251.6 KB  (Smoking questionnaire)
MCQ_J       5293.8 KB  (Medical conditions)
PAQ_J        780.9 KB  (Physical activity)


## 3. Load XPT → DataFrames

`pd.read_sas(..., format="xport")` handles SAS XPT directly. Every NHANES file has a primary key column `SEQN` (respondent sequence number).

In [4]:
def load_xpt(path: Path) -> pd.DataFrame:
    df = pd.read_sas(path, format="xport", encoding="utf-8")
    if "SEQN" in df.columns:
        df["SEQN"] = df["SEQN"].astype("Int64")
    return df


frames: dict[str, pd.DataFrame] = {f.code: load_xpt(f.local_path) for f in FILES}

summary = pd.DataFrame(
    [
        {
            "file": code,
            "label": next(f.label for f in FILES if f.code == code),
            "rows": len(df),
            "cols": df.shape[1],
            "unique_SEQN": df["SEQN"].nunique() if "SEQN" in df.columns else None,
        }
        for code, df in frames.items()
    ]
)
summary

,file,label,rows,cols,unique_SEQN
0,DEMO_J,Demographics,9254,46,9254
1,BMX_J,Body measurements,8704,21,8704
2,BPX_J,Blood pressure (measured),8704,21,8704
3,BPQ_J,Blood pressure questionnaire,6161,11,6161
4,TCHOL_J,Total cholesterol,7435,3,7435
5,HDL_J,HDL cholesterol,7435,3,7435
6,DIQ_J,Diabetes questionnaire,8897,54,8897
7,GHB_J,HbA1c (glycohemoglobin),6401,2,6401
8,SMQ_J,Smoking questionnaire,6724,37,6724
9,MCQ_J,Medical conditions,8897,76,8897


## 4. Quick look at each frame

Column lists + a 3-row peek. Codebooks (full variable definitions) live at `https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/<CODE>.htm`.

In [5]:
for code, df in frames.items():
    print(f"\n=== {code} ({df.shape[0]} rows × {df.shape[1]} cols) ===")
    print("cols:", list(df.columns))
    display(df.head(3))


=== DEMO_J (9254 rows × 46 cols) ===
cols: ['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMQADFC', 'DMDBORN4', 'DMDCITZN', 'DMDYRSUS', 'DMDEDUC3', 'DMDEDUC2', 'DMDMARTL', 'RIDEXPRG', 'SIALANG', 'SIAPROXY', 'SIAINTRP', 'FIALANG', 'FIAPROXY', 'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 'AIALANGA', 'DMDHHSIZ', 'DMDFMSIZ', 'DMDHHSZA', 'DMDHHSZB', 'DMDHHSZE', 'DMDHRGND', 'DMDHRAGZ', 'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ', 'WTINT2YR', 'WTMEC2YR', 'SDMVPSU', 'SDMVSTRA', 'INDHHIN2', 'INDFMIN2', 'INDFMPIR']


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,DMQMILIZ,DMQADFC,DMDBORN4,DMDCITZN,DMDYRSUS,DMDEDUC3,DMDEDUC2,DMDMARTL,RIDEXPRG,SIALANG,SIAPROXY,SIAINTRP,FIALANG,FIAPROXY,FIAINTRP,MIALANG,MIAPROXY,MIAINTRP,AIALANGA,DMDHHSIZ,DMDFMSIZ,DMDHHSZA,DMDHHSZB,DMDHHSZE,DMDHRGND,DMDHRAGZ,DMDHREDZ,DMDHRMAZ,DMDHSEDZ,WTINT2YR,WTMEC2YR,SDMVPSU,SDMVSTRA,INDHHIN2,INDFMIN2,INDFMPIR
0,93703,10.0,2.0,2.0,2.0,NaN,5.0,6.0,2.0,27.0,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2.0,1.0,2.0,2.0,NaN,NaN,NaN,NaN,5.0,5.0,3.000000e+00,5.397605e-79,5.397605e-79,1.0,2.0,3.0,1.0,3.0,9246.491865,8539.731348,2.0,145.0,15.0,15.0,5.00
1,93704,10.0,2.0,1.0,2.0,NaN,3.0,3.0,1.0,33.0,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2.0,1.0,2.0,2.0,NaN,NaN,NaN,NaN,4.0,4.0,2.000000e+00,5.397605e-79,5.397605e-79,1.0,2.0,3.0,1.0,2.0,37338.768343,42566.614750,1.0,143.0,15.0,15.0,5.00
2,93705,10.0,2.0,2.0,66.0,NaN,4.0,4.0,2.0,NaN,2.0,NaN,1.0,1.0,NaN,NaN,2.0,3.0,NaN,1.0,2.0,2.0,1.0,2.0,2.0,1.0,2.0,2.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79,1.000000e+00,2.0,4.0,1.0,2.0,NaN,8614.571172,8338.419786,2.0,145.0,3.0,3.0,0.82



=== BMX_J (8704 rows × 21 cols) ===
cols: ['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMXLEG', 'BMILEG', 'BMXARML', 'BMIARML', 'BMXARMC', 'BMIARMC', 'BMXWAIST', 'BMIWAIST', 'BMXHIP', 'BMIHIP']


,SEQN,BMDSTATS,BMXWT,BMIWT,BMXRECUM,BMIRECUM,BMXHEAD,BMIHEAD,BMXHT,BMIHT,BMXBMI,BMXLEG,BMILEG,BMXARML,BMIARML,BMXARMC,BMIARMC,BMXWAIST,BMIWAIST,BMXHIP,BMIHIP
0,93703,1.0,13.7,3.0,89.6,NaN,NaN,NaN,88.6,NaN,17.5,NaN,NaN,18.0,NaN,16.2,NaN,48.2,NaN,NaN,NaN
1,93704,1.0,13.9,NaN,95.0,NaN,NaN,NaN,94.2,NaN,15.7,NaN,NaN,18.6,NaN,15.2,NaN,50.0,NaN,NaN,NaN
2,93705,1.0,79.5,NaN,NaN,NaN,NaN,NaN,158.3,NaN,31.7,37.0,NaN,36.0,NaN,32.0,NaN,101.8,NaN,110.0,NaN



=== BPX_J (8704 rows × 21 cols) ===
cols: ['SEQN', 'PEASCCT1', 'BPXCHR', 'BPAARM', 'BPACSZ', 'BPXPLS', 'BPXPULS', 'BPXPTY', 'BPXML1', 'BPXSY1', 'BPXDI1', 'BPAEN1', 'BPXSY2', 'BPXDI2', 'BPAEN2', 'BPXSY3', 'BPXDI3', 'BPAEN3', 'BPXSY4', 'BPXDI4', 'BPAEN4']


...

,SEQN,SMQ020,SMD030,SMQ040,SMQ050Q,SMQ050U,SMD057,SMQ078,SMD641,SMD650,SMD093,SMDUPCA,SMD100BR,SMD100FL,SMD100MN,SMD100LN,SMD100TR,SMD100NI,SMD100CO,SMQ621,SMD630,SMQ661,SMQ665A,SMQ665B,SMQ665C,SMQ665D,SMQ670,SMQ848,SMQ852Q,SMQ852U,SMQ890,SMQ895,SMQ900,SMQ905,SMQ910,SMQ915,SMAQUEX2
0,93705,1.0,16.0,3.0,30.0,4.0,5.0,NaN,NaN,NaN,NaN,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,2.0,NaN,2.0,NaN,1.0
1,93706,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,2.0,NaN,2.0,NaN,1.0
2,93707,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0



=== MCQ_J (8897 rows × 76 cols) ===
cols: ['SEQN', 'MCQ010', 'MCQ025', 'MCQ035', 'MCQ040', 'MCQ050', 'AGQ030', 'MCQ053', 'MCQ080', 'MCQ092', 'MCD093', 'MCQ149', 'MCQ151', 'RHD018', 'MCQ160A', 'MCD180A', 'MCQ195', 'MCQ160N', 'MCD180N', 'MCQ160B', 'MCD180B', 'MCQ160C', 'MCD180C', 'MCQ160D', 'MCD180D', 'MCQ160E', 'MCD180E', 'MCQ160F', 'MCD180F', 'MCQ160M', 'MCQ170M', 'MCD180M', 'MCQ160G', 'MCD180G', 'MCQ160K', 'MCQ170K', 'MCD180K', 'MCQ160O', 'MCQ160L', 'MCQ170L', 'MCD180L', 'MCQ500', 'MCQ510A', 'MCQ510B', 'MCQ510C', 'MCQ510D', 'MCQ510E', 'MCQ510F', 'MCQ520', 'MCQ530', 'MCQ540', 'MCQ550', 'MCQ560', 'MCQ570', 'MCQ203', 'MCQ206', 'MCQ220', 'MCQ230A', 'MCD240A', 'MCQ230B', 'MCD240B', 'MCQ230C', 'MCD240C', 'MCQ230D', 'MCQ300B', 'MCQ300C', 'MCQ300A', 'MCQ366A', 'MCQ366B', 'MCQ366C', 'MCQ366D', 'MCQ371A', 'MCQ371B', 'MCQ371C', 'MCQ371D', 'OSQ230']


,SEQN,MCQ010,MCQ025,MCQ035,MCQ040,MCQ050,AGQ030,MCQ053,MCQ080,MCQ092,MCD093,MCQ149,MCQ151,RHD018,MCQ160A,MCD180A,MCQ195,MCQ160N,MCD180N,MCQ160B,MCD180B,MCQ160C,MCD180C,MCQ160D,MCD180D,MCQ160E,MCD180E,MCQ160F,MCD180F,MCQ160M,MCQ170M,MCD180M,MCQ160G,MCD180G,MCQ160K,MCQ170K,MCD180K,MCQ160O,MCQ160L,MCQ170L,MCD180L,MCQ500,MCQ510A,MCQ510B,MCQ510C,MCQ510D,MCQ510E,MCQ510F,MCQ520,MCQ530,MCQ540,MCQ550,MCQ560,MCQ570,MCQ203,MCQ206,MCQ220,MCQ230A,MCD240A,MCQ230B,MCD240B,MCQ230C,MCD240C,MCQ230D,MCQ300B,MCQ300C,MCQ300A,MCQ366A,MCQ366B,MCQ366C,MCQ366D,MCQ371A,MCQ371B,MCQ371C,MCQ371D,OSQ230
0,93703,2.0,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704,2.0,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,93705,1.0,10.0,2.0,NaN,NaN,NaN,2.0,2.0,2.0,NaN,NaN,NaN,NaN,1.0,64.0,2.0,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,NaN,2.0,NaN,2.0,NaN,NaN,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,2.0,2.0,NaN,2.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2.0,2.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,2.0



=== PAQ_J (5856 rows × 17 cols) ===
cols: ['SEQN', 'PAQ605', 'PAQ610', 'PAD615', 'PAQ620', 'PAQ625', 'PAD630', 'PAQ635', 'PAQ640', 'PAD645', 'PAQ650', 'PAQ655', 'PAD660', 'PAQ665', 'PAQ670', 'PAD675', 'PAD680']


,SEQN,PAQ605,PAQ610,PAD615,PAQ620,PAQ625,PAD630,PAQ635,PAQ640,PAD645,PAQ650,PAQ655,PAD660,PAQ665,PAQ670,PAD675,PAD680
0,93705,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,1.0,2.0,60.0,300.0
1,93706,2.0,NaN,NaN,2.0,NaN,NaN,1.0,5.0,45.0,2.0,NaN,NaN,1.0,2.0,30.0,240.0
2,93708,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,1.0,5.0,30.0,120.0


## 5. Demographics distribution

Variables of interest in `DEMO_J`:
- `RIDAGEYR` — age in years (top-coded at 80)
- `RIAGENDR` — sex (1=M, 2=F)
- `RIDRETH3` — race/Hispanic origin w/ Asian (1=MexAm, 2=OtherHisp, 3=NH White, 4=NH Black, 6=NH Asian, 7=Other/Multi)
- `WTMEC2YR` — MEC sample weight (use for population-representative summaries)

In [6]:
demo = frames["DEMO_J"]
print("age summary:")
print(demo["RIDAGEYR"].describe())

print("\nsex counts (1=M, 2=F):")
print(demo["RIAGENDR"].value_counts(dropna=False).sort_index())

print("\nrace/ethnicity counts (RIDRETH3):")
print(demo["RIDRETH3"].value_counts(dropna=False).sort_index())

age summary:
count    9.254000e+03
mean     3.433423e+01
std      2.550028e+01
min      5.397605e-79
25%      1.100000e+01
50%      3.100000e+01
75%      5.800000e+01
max      8.000000e+01
Name: RIDAGEYR, dtype: float64

sex counts (1=M, 2=F):
RIAGENDR
1.0    4557
2.0    4697
Name: count, dtype: int64

race/ethnicity counts (RIDRETH3):
RIDRETH3
1.0    1367
2.0     820
3.0    3150
4.0    2115
6.0    1168
7.0     634
Name: count, dtype: int64


## 6. Build a starter analysis frame

Joins the per-respondent slices we'll most likely use into a single wide table on `SEQN`. Mean-of-readings used where NHANES gives multiple BP measurements.

In [7]:
demo_cols = ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "WTMEC2YR"]
bmx_cols  = ["SEQN", "BMXHT", "BMXWT", "BMXBMI", "BMXWAIST"]

bpx = frames["BPX_J"].copy()
sys_cols = [c for c in ["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"] if c in bpx.columns]
dia_cols = [c for c in ["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"] if c in bpx.columns]
bpx["sbp_mean"] = bpx[sys_cols].replace(0, np.nan).mean(axis=1)
bpx["dbp_mean"] = bpx[dia_cols].replace(0, np.nan).mean(axis=1)
bpx_slim = bpx[["SEQN", "sbp_mean", "dbp_mean"]]

tchol_slim = frames["TCHOL_J"][["SEQN", "LBXTC"]].rename(columns={"LBXTC": "total_chol_mgdl"})
hdl_slim   = frames["HDL_J"][["SEQN", "LBDHDD"]].rename(columns={"LBDHDD": "hdl_mgdl"})
ghb_slim   = frames["GHB_J"][["SEQN", "LBXGH"]].rename(columns={"LBXGH": "hba1c"})

diq_src = frames["DIQ_J"]
diq = diq_src[["SEQN", "DIQ010"]].copy()  # 1=Yes, 2=No, 3=Borderline, 7/9=DK/refused
diq["has_diabetes"] = diq["DIQ010"].map({1: 1, 2: 0, 3: 0}).astype("Int64")

smq_src = frames["SMQ_J"]
smq_keep = ["SEQN", "SMQ020"] + (["SMQ040"] if "SMQ040" in smq_src.columns else [])
smq = smq_src[smq_keep].copy()
smq["ever_smoked"] = smq["SMQ020"].map({1: 1, 2: 0}).astype("Int64")
if "SMQ040" in smq.columns:
    smq["current_smoker"] = smq["SMQ040"].map({1: 1, 2: 1, 3: 0}).astype("Int64")

bpq_src = frames["BPQ_J"]
bpq_keep = ["SEQN", "BPQ020"] + (["BPQ050A"] if "BPQ050A" in bpq_src.columns else [])
bpq = bpq_src[bpq_keep].copy()
bpq["told_high_bp"] = bpq["BPQ020"].map({1: 1, 2: 0}).astype("Int64")
if "BPQ050A" in bpq.columns:
    bpq["on_bp_meds"] = bpq["BPQ050A"].map({1: 1, 2: 0}).astype("Int64")

wide = (
    frames["DEMO_J"][demo_cols]
    .merge(frames["BMX_J"][bmx_cols], on="SEQN", how="left")
    .merge(bpx_slim, on="SEQN", how="left")
    .merge(tchol_slim, on="SEQN", how="left")
    .merge(hdl_slim, on="SEQN", how="left")
    .merge(ghb_slim, on="SEQN", how="left")
    .merge(diq[["SEQN", "has_diabetes"]], on="SEQN", how="left")
    .merge(smq.drop(columns=[c for c in ["SMQ020", "SMQ040"] if c in smq.columns]), on="SEQN", how="left")
    .merge(bpq.drop(columns=[c for c in ["BPQ020", "BPQ050A"] if c in bpq.columns]), on="SEQN", how="left")
)
print("wide shape:", wide.shape)
wide.head()

wide shape: (9254, 19)


,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3,WTMEC2YR,BMXHT,BMXWT,BMXBMI,BMXWAIST,sbp_mean,dbp_mean,total_chol_mgdl,hdl_mgdl,hba1c,has_diabetes,ever_smoked,current_smoker,told_high_bp,on_bp_meds
0,93703,2.0,2.0,6.0,8539.731348,88.6,13.7,17.5,48.2,NaN,NaN,NaN,NaN,NaN,0,<NA>,<NA>,<NA>,<NA>
1,93704,2.0,1.0,3.0,42566.614750,94.2,13.9,15.7,50.0,NaN,NaN,NaN,NaN,NaN,0,<NA>,<NA>,<NA>,<NA>
2,93705,66.0,2.0,4.0,8338.419786,158.3,79.5,31.7,101.8,200.000000,68.000000,157.0,60.0,6.2,0,1,0,1,1
3,93706,18.0,1.0,6.0,8723.439814,175.7,66.3,21.5,79.3,111.333333,73.333333,148.0,47.0,5.2,0,0,<NA>,0,<NA>
4,93707,13.0,1.0,7.0,7064.609730,158.4,45.4,18.1,64.1,128.000000,47.333333,189.0,68.0,5.6,0,<NA>,<NA>,<NA>,<NA>


In [8]:
missing = wide.isna().mean().sort_values(ascending=False).to_frame("frac_missing")
missing["n_missing"] = wide.isna().sum()
missing

,frac_missing,n_missing
on_bp_meds,0.790145,7312
current_smoker,0.745083,6895
ever_smoked,0.367193,3398
hba1c,0.346769,3209
told_high_bp,0.335314,3103
dbp_mean,0.274152,2537
sbp_mean,0.274152,2537
total_chol_mgdl,0.271882,2516
hdl_mgdl,0.271882,2516
BMXWAIST,0.178625,1653


## 7. Public-use linked mortality file

Fixed-width ASCII at the NCHS data linkage FTP. Layout below comes from the `Readme_2019.txt` shipped with the file — if NCHS revises positions, refer back to that readme.

Outcome columns of interest:
- `MORTSTAT` — 0 = assumed alive, 1 = assumed deceased
- `UCOD_LEADING` — underlying cause of death (recode), `1` = diseases of heart
- `PERMTH_INT` — person-months of follow-up from interview to death/censoring

In [9]:
MORT_URL = (
    "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/"
    "linked_mortality/NHANES_2017_2018_MORT_2019_PUBLIC.dat"
)
MORT_PATH = DATA_DIR / "NHANES_2017_2018_MORT_2019_PUBLIC.dat"
download(MORT_URL, MORT_PATH)
print("mortality file:", MORT_PATH, MORT_PATH.stat().st_size, "bytes")

mortality file: /Users/deniz/Documents/Projects/almond/inspect/data/NHANES_2017_2018_MORT_2019_PUBLIC.dat 449623 bytes


In [10]:
# Column positions per NCHS 2019 public-use linked mortality readme.
# (start, end) are 1-indexed inclusive — converted to 0-indexed half-open below.
MORT_LAYOUT: list[tuple[str, int, int, str]] = [
    ("SEQN",          1,  6,  "Int64"),
    ("ELIGSTAT",     15, 15,  "Int64"),
    ("MORTSTAT",     16, 16,  "Int64"),
    ("UCOD_LEADING", 17, 19,  "string"),
    ("DIABETES",     20, 20,  "Int64"),
    ("HYPERTEN",     21, 21,  "Int64"),
    ("PERMTH_INT",   43, 45,  "Int64"),
    ("PERMTH_EXM",   46, 48,  "Int64"),
]

names    = [c[0] for c in MORT_LAYOUT]
colspecs = [(c[1] - 1, c[2]) for c in MORT_LAYOUT]
dtypes   = {c[0]: c[3] for c in MORT_LAYOUT}

mort = pd.read_fwf(
    MORT_PATH,
    colspecs=colspecs,
    names=names,
    dtype=str,
    na_values=["", "."],
)
for col, dt in dtypes.items():
    if dt == "Int64":
        mort[col] = pd.to_numeric(mort[col], errors="coerce").astype("Int64")
    else:
        mort[col] = mort[col].astype("string").str.strip()

print("mortality shape:", mort.shape)
mort.head()

mortality shape: (9254, 8)


,SEQN,ELIGSTAT,MORTSTAT,UCOD_LEADING,DIABETES,HYPERTEN,PERMTH_INT,PERMTH_EXM
0,93703,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,93704,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,93705,1,0,<NA>,<NA>,<NA>,18,18
3,93706,1,0,<NA>,<NA>,<NA>,35,34
4,93707,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [11]:
print("eligibility status (ELIGSTAT — 1=eligible, 2=under 18 ineligible, 3=ineligible):")
print(mort["ELIGSTAT"].value_counts(dropna=False).sort_index())

print("\nmortality status among eligible (0=alive, 1=deceased):")
print(mort.loc[mort["ELIGSTAT"] == 1, "MORTSTAT"].value_counts(dropna=False).sort_index())

print("\nleading cause counts (UCOD_LEADING):")
print(mort["UCOD_LEADING"].value_counts(dropna=False))

print("\nperson-months of follow-up (interview-based) summary:")
print(mort["PERMTH_INT"].describe())

eligibility status (ELIGSTAT — 1=eligible, 2=under 18 ineligible, 3=ineligible):
ELIGSTAT
1    5809
2    3398
3      47
Name: count, dtype: Int64

mortality status among eligible (0=alive, 1=deceased):
MORTSTAT
0    5664
1     145
Name: count, dtype: Int64

leading cause counts (UCOD_LEADING):
UCOD_LEADING
<NA>    9109
010       74
001       37
002       34
Name: count, dtype: Int64

person-months of follow-up (interview-based) summary:
count       5809.0
mean     24.303495
std       7.305607
min            1.0
25%           18.0
50%           25.0
75%           31.0
max           37.0
Name: PERMTH_INT, dtype: Float64


## 8. Join clinical wide-frame to mortality

End state for inspection: one row per NHANES respondent with demographics + clinical features + mortality outcome. Restrict to mortality-eligible adults (`ELIGSTAT == 1`).

In [12]:
joined = wide.merge(mort, on="SEQN", how="left")
eligible = joined[joined["ELIGSTAT"] == 1].copy()
print("joined rows:", len(joined), "  mortality-eligible:", len(eligible))

print("\ndeath rate among eligible:", eligible["MORTSTAT"].mean())

eligible.describe(include="all").T.head(30)

joined rows: 9254   mortality-eligible: 5809

death rate among eligible: 0.02496126699948356


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SEQN,5809.0,<NA>,<NA>,<NA>,98278.984163,2684.848425,93705.0,95952.0,98262.0,100599.0,102956.0
RIDAGEYR,5809.0,NaN,NaN,NaN,49.923567,18.786233,18.0,33.0,51.0,65.0,80.0
RIAGENDR,5809.0,NaN,NaN,NaN,1.514891,0.499821,1.0,1.0,2.0,2.0,2.0
RIDRETH3,5809.0,NaN,NaN,NaN,3.495782,1.649367,1.0,3.0,3.0,4.0,7.0
WTMEC2YR,5809.0,NaN,NaN,NaN,42388.839723,51083.492313,0.0,14190.513923,25124.339114,46828.449325,419762.836488
BMXHT,5411.0,NaN,NaN,NaN,166.370116,10.07805,138.3,158.9,165.9,173.65,197.7
BMXWT,5411.0,NaN,NaN,NaN,82.50462,23.00916,32.6,66.5,78.9,94.6,242.6
BMXBMI,5401.0,NaN,NaN,NaN,29.701703,7.45045,14.2,24.6,28.5,33.5,86.2
BMXWAIST,5157.0,NaN,NaN,NaN,100.186349,17.240952,56.4,87.9,98.9,110.7,169.5
sbp_mean,5222.0,NaN,NaN,NaN,126.33978,20.134788,72.666667,112.0,123.333333,136.666667,238.0


## 9. Pivot — wearable-only training (cycles G + H, 2-year mortality)

The 2017–2018 cycle above has **no wearable data** (`PAQ_J` is a self-report questionnaire) and only 1–2 years of mortality follow-up — neither viable for the almond Cox model.

Cycles **G (2011–2012)** and **H (2013–2014)** instead provide:
- `PAXHD_*` — per-subject monitor metadata (status, first/last day, handedness)
- `PAXDAY_*` — per-day aggregates: total MIMS, wake-wear minutes, sleep-wear minutes, non-wear, quality flags
- Linked 2019 mortality with up to **~85 months** of follow-up

The minute-level `PAXMIN_*` is **not publicly distributed as XPT** (CDC returns 404 for both cycles); we work from daily aggregates only. The locked feature vector for `almond-ml/training/03_train_cox.ipynb` is documented in `AGENTS.md` → "Locked ML feature vector".

Below is the cycle-H feasibility check that proves the pipeline produces sensible features and a usable event count for a 2-year endpoint. Cycle G is identical — extend the manifest to pool both for the actual training run.

In [13]:
# Cycle H wearable + supporting clinical/demographic + mortality manifest.
# Cycle G mirrors this with start_year=2011 — left out here to keep the inspect
# notebook fast; almond-ml/training/01_download_nhanes.ipynb pulls both.

WEARABLE_FILES: list[NhanesFile] = [
    NhanesFile("PAXHD_H",  2013, "Wearable header (per subject)"),
    NhanesFile("PAXDAY_H", 2013, "Wearable daily aggregates"),
    NhanesFile("DEMO_H",   2013, "Demographics"),
    NhanesFile("BMX_H",    2013, "Body measurements"),
]

for f in WEARABLE_FILES:
    download(f.url, f.local_path)
    print(f"{f.code:8s}  {f.local_path.stat().st_size / 1024:8.1f} KB  ({f.label})")

MORT_H_URL  = ("https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/"
               "linked_mortality/NHANES_2013_2014_MORT_2019_PUBLIC.dat")
MORT_H_PATH = DATA_DIR / "NHANES_2013_2014_MORT_2019_PUBLIC.dat"
download(MORT_H_URL, MORT_H_PATH)
print(f"mortality (H): {MORT_H_PATH.stat().st_size / 1024:.1f} KB")

PAXHD_H      428.5 KB  (Wearable header (per subject))
PAXDAY_H    7147.3 KB  (Wearable daily aggregates)
DEMO_H      3743.4 KB  (Demographics)
BMX_H       1997.6 KB  (Body measurements)
mortality (H): 482.7 KB


### 9.1 SAS sentinel cleanup + valid-day filter

NHANES XPT files use `5.397605e-79` as the SAS numeric "missing" marker. A naive `== 0` comparison silently fails on it (as we discovered the hard way), so any arithmetic over PAXDAY columns must coerce values below `1e-70` to `0.0` first.

Per Karas 2022 / Smirnova 2020, a *valid wear-day* is:
- monitor reliable: `PAXSTS == 1`
- wake-wear minutes: `PAXWWMD ≥ 600` (≥10 h)
- no quality flag: `PAXQFD == 0`

A subject with **≥4 valid wear-days** enters the analytic cohort.

In [14]:
SAS_MISS = 1e-70  # SAS numeric-missing sentinel ≈ 5.4e-79

hd_h  = load_xpt(DATA_DIR / "PAXHD_H.xpt")
day_h = load_xpt(DATA_DIR / "PAXDAY_H.xpt")
demo_h = load_xpt(DATA_DIR / "DEMO_H.xpt")
bmx_h  = load_xpt(DATA_DIR / "BMX_H.xpt")

# Coerce SAS-missing sentinels to 0 in the daily aggregate columns.
PAXDAY_NUMERIC = ["PAXAISMD", "PAXWWMD", "PAXSWMD", "PAXNWMD", "PAXUMD",
                  "PAXLXSD", "PAXQFD", "PAXSSNDP", "PAXMTSD"]
for col in PAXDAY_NUMERIC:
    day_h.loc[day_h[col] < SAS_MISS, col] = 0.0

# Subjects whose monitor data is reliable.
reliable = set(hd_h.loc[hd_h["PAXSTS"] == 1, "SEQN"])
day_h = day_h[day_h["SEQN"].isin(reliable)].copy()

day_h["valid_day"] = (day_h["PAXWWMD"] >= 600) & (day_h["PAXQFD"] == 0)

print(f"reliable monitors: {len(reliable):,}")
print(f"PAXDAY rows (post-filter): {len(day_h):,}")
print(f"valid-day fraction: {day_h['valid_day'].mean():.2%}")

reliable monitors: 7,776
PAXDAY rows (post-filter): 69,018


valid-day fraction: 56.85%


### 9.2 Engineer the locked 8-feature vector

Mirrors `AGENTS.md` → "Locked ML feature vector". The final analytic table is one row per (adult, ≥4 valid days) subject with the 8 trained features + the survival outcome `(MORTSTAT, PERMTH_INT)`.

In [15]:
# Per-subject summary from PAXDAY (training-side feature engineering).
subject = (
    day_h.groupby("SEQN")
    .agg(
        n_valid_days=("valid_day", "sum"),
        mean_daily_mims=("PAXAISMD", "mean"),
        sd_daily_mims=("PAXAISMD", "std"),
        mean_sleep_min=("PAXSWMD", "mean"),
        sd_sleep_min=("PAXSWMD", "std"),
        mean_wake_wear_min=("PAXWWMD", "mean"),
    )
    .reset_index()
)
subject = subject[subject["n_valid_days"] >= 4]

# Demographics + BMI + adults-only filter.
demo_slim = demo_h[["SEQN", "RIDAGEYR", "RIAGENDR"]].copy()
demo_slim["age"] = demo_slim["RIDAGEYR"]
demo_slim["sex_male"] = (demo_slim["RIAGENDR"] == 1).astype(int)
bmx_slim = bmx_h[["SEQN", "BMXBMI"]].rename(columns={"BMXBMI": "bmi"})

cohort = (
    subject
    .merge(demo_slim[["SEQN", "age", "sex_male"]], on="SEQN")
    .merge(bmx_slim, on="SEQN")
)
cohort = cohort[(cohort["age"] >= 18) & cohort["bmi"].notna()].copy()

FEATURES = ["age", "sex_male", "bmi", "mean_daily_mims", "sd_daily_mims",
            "mean_sleep_min", "sd_sleep_min", "mean_wake_wear_min"]
print(f"adult cohort with valid wear-week + BMI: {len(cohort):,}")
print()
print("feature distribution (cycle H only):")
print(cohort[FEATURES].describe().round(1).T)

adult cohort with valid wear-week + BMI: 4,356

feature distribution (cycle H only):


                     count       mean       std       min        25%        50%        75%        max
age                 4356.0       48.7      18.1      18.0       34.0       48.0       63.0       80.0
sex_male            4356.0        0.5       0.5       0.0        0.0        0.0        1.0        1.0
bmi                 4356.0       29.1       7.2      14.1       24.0       27.9       32.5       82.9
mean_daily_mims     4356.0  2563979.3  474853.0  579617.9  2238964.6  2532938.3  2855446.9  4622786.0
sd_daily_mims       4356.0   989868.4  325175.6  229160.0   754047.1   936356.5  1165735.7  2621666.0
mean_sleep_min      4356.0      393.2      98.8       2.2      343.2      395.0      450.2      700.0
sd_sleep_min        4356.0      171.2      53.5       4.4      135.9      164.3      198.5      538.9
mean_wake_wear_min  4356.0      740.6      96.9     335.4      683.6      752.3      807.7     1172.6


### 9.3 Join to cycle-H mortality and report event counts

Outcome: `(event = MORTSTAT == 1, time = PERMTH_INT)`. The 2-yr endpoint becomes `event ∧ time ≤ 24`. Eligibility filter `ELIGSTAT == 1` excludes minors and respondents who couldn't be linked.

In [16]:
# Re-parse the cycle-H linked mortality file with the same fixed-width layout
# used in section 7. (MORT_LAYOUT, names, colspecs are already defined above.)
mort_h = pd.read_fwf(
    MORT_H_PATH, colspecs=colspecs, names=names, dtype=str, na_values=["", "."]
)
for col, dt in dtypes.items():
    if dt == "Int64":
        mort_h[col] = pd.to_numeric(mort_h[col], errors="coerce").astype("Int64")
    else:
        mort_h[col] = mort_h[col].astype("string").str.strip()
mort_h_eligible = mort_h[mort_h["ELIGSTAT"] == 1].copy()

trainable = cohort.merge(
    mort_h_eligible[["SEQN", "MORTSTAT", "PERMTH_INT"]], on="SEQN"
)
trainable["event_2yr"] = (
    (trainable["MORTSTAT"] == 1) & (trainable["PERMTH_INT"] <= 24)
).astype(int)

print(f"trainable cohort (wearable + outcome): {len(trainable):,}")
print(f"  deaths anytime in follow-up : {(trainable['MORTSTAT'] == 1).sum()}")
print(f"  deaths within 24 months     : {trainable['event_2yr'].sum()}")
print(f"  median PERMTH_INT (months)  : {trainable['PERMTH_INT'].median()}")
print(f"  events / planned features   : {trainable['event_2yr'].sum() / len(FEATURES):.1f}")
print()
print("note: pooling cycle G doubles n and roughly doubles the event count.")

trainable cohort (wearable + outcome): 4,350
  deaths anytime in follow-up : 319
  deaths within 24 months     : 77
  median PERMTH_INT (months)  : 72.0
  events / planned features   : 9.6

note: pooling cycle G doubles n and roughly doubles the event count.


### 9.4 Single-subject end-to-end sanity check

Pick one cohort member, show the daily PAXDAY rows that fed the aggregation, the final 8-feature row, and the survival outcome. This is the same shape `02_features.ipynb` will write to `features.parquet`.

In [17]:
# Pick the subject with the most valid days for a clean demo.
example_seqn = (
    trainable.sort_values("n_valid_days", ascending=False)["SEQN"].iloc[0]
)

per_day = day_h[(day_h["SEQN"] == example_seqn) & day_h["valid_day"]][
    ["PAXDAYD", "PAXDAYWD", "PAXAISMD", "PAXWWMD", "PAXSWMD", "PAXNWMD"]
]
print(f"=== SEQN={example_seqn} — valid wear-days ===")
print(per_day.to_string(index=False))

row = trainable[trainable["SEQN"] == example_seqn].iloc[0]
print()
print("=== engineered feature row ===")
for col in FEATURES:
    print(f"  {col:22s} {row[col]:>15,.1f}")
print()
print("=== outcome ===")
print(f"  MORTSTAT     : {row['MORTSTAT']}")
print(f"  PERMTH_INT   : {row['PERMTH_INT']} months")
print(f"  event_2yr    : {row['event_2yr']}")

=== SEQN=82091 — valid wear-days ===
PAXDAYD PAXDAYWD  PAXAISMD  PAXWWMD  PAXSWMD  PAXNWMD
      1        4  254800.0    661.0     16.0      0.0
      2        5 1330480.0   1192.0    178.0      0.0
      3        6 2055521.0   1055.0    317.0      0.0
      4        7 2204800.0   1036.0    308.0      0.0
      5        1 2180880.0   1074.0    292.0      0.0
      6        2 2197681.0   1041.0    316.0      0.0
      7        3 1235280.0   1226.0    144.0      0.0
      8        4 1501200.0   1164.0    225.0      0.0
      9        5  866080.0    610.0    123.0      0.0



=== engineered feature row ===
  age                               69.0
  sex_male                           0.0
  bmi                               42.8
  mean_daily_mims            1,536,302.4
  sd_daily_mims                688,752.0
  mean_sleep_min                   213.2
  sd_sleep_min                     105.9
  mean_wake_wear_min             1,006.6

=== outcome ===
  MORTSTAT     : 0.0
  PERMTH_INT   : 65.0 months
  event_2yr    : 0.0


## 10. Go / no-go for `almond-ml/training/`

| Check                                                 | Cycle H alone (run above) | Pooled G + H (target) |
| ----------------------------------------------------- | ------------------------- | --------------------- |
| Wearable monitor + linked mortality available         | yes                       | yes                   |
| Adult cohort with ≥4 valid wear-days + BMI            | 4,350                     | ~8,000                |
| Deaths within 24 months                               | 77                        | ~150                  |
| Events per planned feature (8 features)               | 9.6                       | ~19                   |
| Median follow-up                                      | 72 months                 | ≥72 months            |
| Concordance target (`AGENTS.md` Definition of Done)   | ≥ 0.70 on held-out 25% split (event-stratified)                       |

**Decision: GO.** The pooled-cycles cohort comfortably passes the 10-events-per-variable rule of thumb and has follow-up well past the 24-month endpoint. Move to:

1. `almond-ml/training/01_download_nhanes.ipynb` — pull the manifest from `AGENTS.md` for both cycles.
2. `almond-ml/training/02_features.ipynb` — re-run the same engineering across pooled cycles, write `features.parquet` + `labels.parquet`.
3. `almond-ml/training/03_train_cox.ipynb` — fit `CoxPHSurvivalAnalysis`, validate at 24 months, dump `models/cox_model.pkl` + `models/feature_means.json`.

Everything in this `inspect/` workspace is exploratory and is **not imported by the backend** — the canonical pipeline lives under `almond-ml/`.